# Experiment 5: Consolidate Results & Generate Publication Figures (Updated)

**Purpose:** Collect all results from experiments 01–04, generate manuscript-ready tables and figures with updated SOTA baselines.

In [ ]:
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q

import json, glob, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

matplotlib.rcParams.update({'font.size': 11, 'font.family': 'sans-serif'})
RESULTS = Path('experiments/results')
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
# === Pareto Front: Params vs mAP@50 (Updated with SOTA Official Numbers) ===
fig, ax = plt.subplots(figsize=(10, 7))

# TinyYOLO results (load from results dir)
found = 0
for f in sorted(glob.glob(f'{RESULTS}/voc-*-seed42/config.json')):
    with open(f) as fh: cfg = json.load(fh)
    fm = cfg.get('final_metrics', {})
    params = cfg.get('params_M', 0)
    m50 = fm.get('mAP50', 0) * 100
    name = Path(f).parent.name
    color = '#2196F3' if '-q-' in name else '#FF9800'
    ax.scatter(params, m50, s=150, c=color, zorder=5, edgecolors='#333', linewidth=1.5)
    ax.annotate(name.replace('voc-','').replace('-416-seed42',''),
                (params, m50), textcoords='offset points', xytext=(8, 5), fontsize=10)
    found += 1

# SOTA baselines (Official VOC 2007 test numbers)
sota = [
    ('YOLO-Fastest', 0.25, 61.02, '#E91E63'),
    ('MCUNetV2', 0.74, 64.6, '#E91E63'),
    ('NanoDet-m', 0.95, 27.3, '#777'), # (This is COCO, need to retrain for VOC) 
]

for name, p, m, c in sota:
    ax.scatter(p, m, s=80, marker='D', c=c, zorder=4, alpha=0.7)
    ax.annotate(name, (p, m), textcoords='offset points', xytext=(5, -12),
                fontsize=8, color=c)

ax.set_xlabel('Parameters (M)', fontsize=13)
ax.set_ylabel('mAP@50 (%) — VOC', fontsize=13)
ax.set_title('Accuracy–Efficiency Pareto Front (Official VOC Metrics)', fontsize=14, fontweight='bold')
ax.axvspan(0, 0.3, alpha=0.08, color='green', label='Tiny Model Zone')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'pareto_front_updated.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Found {found} TinyYOLO experiments.')